In [1]:
import numpy as np
import pandas as pd
from arch import arch_model

In [2]:
# Step 1: Loading Phase 2 outputs

In [3]:
portfolio_dollar_value = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_dollar_value.csv', index_col =0, parse_dates =True).iloc[:,0]
portfolio_pnl = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_pnl.csv', index_col =0, parse_dates =True).iloc[:,0]

In [4]:
port_simple_returns = portfolio_dollar_value.pct_change().dropna()

In [5]:
port_log_returns = np.log(portfolio_dollar_value / portfolio_dollar_value.shift(1)).dropna()

In [6]:
log_returns_pct = port_log_returns * 100 

In [7]:
# Step 2: Fitting GJR-GARCH on an initial training window only

In [9]:
TRAIN_WINDOW = 750
 
garch_train_returns = log_returns_pct.iloc[:TRAIN_WINDOW]
gjr_spec = arch_model(garch_train_returns, vol='GARCH', p=1, o=1, q=1, dist='normal', mean='Constant')
gjr_fit = gjr_spec.fit(disp='off')
 
mu = gjr_fit.params['mu']
omega = gjr_fit.params['omega']
alpha = gjr_fit.params['alpha[1]']
gamma = gjr_fit.params['gamma[1]']   # the leverage/asymmetry parameter
beta = gjr_fit.params['beta[1]']
 
print(f"\nFitted GJR-GARCH(1,1) parameters (frozen for the whole backtest):")
print(f"  mu={mu:.6f}  omega={omega:.6f}  alpha={alpha:.4f}  gamma={gamma:.4f}  beta={beta:.4f}")
print(f"  (alpha+beta+gamma/2={alpha + beta + gamma / 2:.4f} - must be < 1 for stationarity)")
 
assert alpha + beta + gamma / 2 < 1, (
    f"Non-stationary GJR-GARCH fit - the model has no mean-reverting long-run variance. "
)

assert gamma >= 0, (
    f"Negative leverage parameter (gamma={gamma:.4f}) "
)
 


Fitted GJR-GARCH(1,1) parameters (frozen for the whole backtest):
  mu=0.045519  omega=0.067476  alpha=0.0000  gamma=0.2013  beta=0.8600
  (alpha+beta+gamma/2=0.9606 - must be < 1 for stationarity)


In [10]:
# Step 3: Building the full conditional volatility series

In [11]:
n = len(log_returns_pct)
gjr_variance_pct = np.full(n, np.nan)
gjr_variance_pct[:TRAIN_WINDOW] = gjr_fit.conditional_volatility.values ** 2
 
log_returns_pct_array = log_returns_pct.values
for t in range(TRAIN_WINDOW, n):
    resid_prev = log_returns_pct_array[t - 1] - mu
    leverage_term = gamma * (resid_prev ** 2) if resid_prev < 0 else 0.0
    gjr_variance_pct[t] = omega + alpha * (resid_prev ** 2) + leverage_term + beta * gjr_variance_pct[t - 1]
 
gjr_vol = pd.Series(np.sqrt(gjr_variance_pct) / 100, index=port_log_returns.index)
 
standardized_residuals = (port_log_returns - mu / 100) / gjr_vol
standardized_residuals.name = 'GJR_GARCH_Standardized_Residuals'

In [12]:
# Step 4: Aligning to a common date index

In [13]:
common_idx = port_simple_returns.index.intersection(gjr_vol.index)
 
assert len(common_idx) > 0, (
    "No overlapping dates between port_simple_returns and gjr_vol. "
)
 
port_simple_returns_aligned = port_simple_returns.loc[common_idx]
gjr_vol_aligned = gjr_vol.loc[common_idx]
 
print(f"\nTotal available trading days for GJR-GARCH engine: {len(port_simple_returns_aligned)}")


Total available trading days for GJR-GARCH engine: 1506


In [14]:
# Step 5: Parameters and Core Risk Function 

In [15]:
windows = [250, 500, 750]
confidence_levels = [0.95, 0.99, 0.995]
MAX_SCALING_RATIO = 3.0

def calculate_historical_var_es(returns_window, conf_level):
    percentile = (1 - conf_level) * 100
    var_pct = -np.percentile(returns_window, percentile, method='lower')
    tail_returns = returns_window[returns_window <= -var_pct]
    es_pct = -tail_returns.mean() if len(tail_returns) > 0 else var_pct
    return var_pct, es_pct

In [16]:
# Step 6: Initializing the Rolling Engine

In [17]:
gjr_metrics = pd.DataFrame(index=port_simple_returns_aligned.index)
gjr_metrics['Portfolio_Dollar_Value'] = portfolio_dollar_value.loc[port_simple_returns_aligned.index]
gjr_metrics['Daily_PnL'] = portfolio_pnl.loc[port_simple_returns_aligned.index]
gjr_metrics['Portfolio_Dollar_Value_Prior'] = gjr_metrics['Portfolio_Dollar_Value'].shift(1)
gjr_metrics['GJR_GARCH_Vol_Daily'] = gjr_vol_aligned
gjr_metrics['GJR_GARCH_Standardized_Residuals'] = standardized_residuals.loc[port_simple_returns_aligned.index]
 
ret_array = port_simple_returns_aligned.values
vol_array = gjr_vol_aligned.values

In [18]:
# Step 7: Executing Rolling Volatility Scaled Calculations

In [19]:
for w in windows:
    for cl in confidence_levels:
        print(f"Calculating GJR-GARCH-Scaled {w}-day window at {cl * 100}% confidence...")
 
        var_list = np.full(len(ret_array), np.nan)
        es_list = np.full(len(ret_array), np.nan)
 
        for i in range(w, len(ret_array)):
            window_returns = ret_array[i - w:i]
            window_vols = vol_array[i - w:i]
            current_vol_forecast = vol_array[i]
 
            scaling_ratios = current_vol_forecast / window_vols
            scaling_ratios = np.clip(scaling_ratios, a_min=None, a_max=MAX_SCALING_RATIO)
            scaled_window_returns = window_returns * scaling_ratios
 
            v, e = calculate_historical_var_es(scaled_window_returns, cl)
            var_list[i] = v
            es_list[i] = e
 
        col_base = f"{w}d_{cl * 100}%"
        gjr_metrics[f"GJR_VaR_pct_{col_base}"] = var_list
        gjr_metrics[f"GJR_ES_pct_{col_base}"] = es_list
 
        gjr_metrics[f"GJR_VaR_1d_$_" + col_base] = (
            gjr_metrics[f"GJR_VaR_pct_{col_base}"] * gjr_metrics['Portfolio_Dollar_Value_Prior']
        )
        gjr_metrics[f"GJR_ES_1d_$_" + col_base] = (
            gjr_metrics[f"GJR_ES_pct_{col_base}"] * gjr_metrics['Portfolio_Dollar_Value_Prior']
        )
        gjr_metrics[f"GJR_VaR_10d_$_" + col_base] = gjr_metrics[f"GJR_VaR_1d_$_" + col_base] * np.sqrt(10)
        gjr_metrics[f"GJR_ES_10d_$_" + col_base] = gjr_metrics[f"GJR_ES_1d_$_" + col_base] * np.sqrt(10)

Calculating GJR-GARCH-Scaled 250-day window at 95.0% confidence...
Calculating GJR-GARCH-Scaled 250-day window at 99.0% confidence...
Calculating GJR-GARCH-Scaled 250-day window at 99.5% confidence...
Calculating GJR-GARCH-Scaled 500-day window at 95.0% confidence...
Calculating GJR-GARCH-Scaled 500-day window at 99.0% confidence...
Calculating GJR-GARCH-Scaled 500-day window at 99.5% confidence...
Calculating GJR-GARCH-Scaled 750-day window at 95.0% confidence...
Calculating GJR-GARCH-Scaled 750-day window at 99.0% confidence...
Calculating GJR-GARCH-Scaled 750-day window at 99.5% confidence...


In [20]:
# Step 8: Finalizing the Out-of-Sample Dataset

In [21]:
out_of_sample_gjr = gjr_metrics.dropna(subset=['GJR_VaR_1d_$_750d_99.0%'])
 
stray_nan_count = out_of_sample_gjr.isna().sum().sum()
if stray_nan_count > 0:
    print(f"WARNING: {stray_nan_count} unexpected NaN value(s) remain outside the checked column.")
else:
    print("No stray NaN values found elsewhere in the out-of-sample dataset.")
 
print(f"\nPhase 7 Complete. Out-of-sample backtesting days available: {len(out_of_sample_gjr)}")

No stray NaN values found elsewhere in the out-of-sample dataset.

Phase 7 Complete. Out-of-sample backtesting days available: 756


In [22]:
# Step 9: Validations (checked across ALL out-of-sample days)

In [23]:
var_1d = out_of_sample_gjr['GJR_VaR_1d_$_750d_99.0%']
var_10d = out_of_sample_gjr['GJR_VaR_10d_$_750d_99.0%']
es_1d = out_of_sample_gjr['GJR_ES_1d_$_750d_99.0%']
 
assert (var_1d > 0).all(), "Validation Failed: GJR VaR must be positive on every day."
assert (var_10d > var_1d).all(), "Validation Failed: 10-day GJR VaR must exceed 1-day on every day."
assert (es_1d >= var_1d).all(), "Validation Failed: GJR ES must be >= GJR VaR on every day."
 
var_95 = out_of_sample_gjr['GJR_VaR_1d_$_750d_95.0%']
var_99 = out_of_sample_gjr['GJR_VaR_1d_$_750d_99.0%']
var_995 = out_of_sample_gjr['GJR_VaR_1d_$_750d_99.5%']
 
assert (var_99 >= var_95).all(), "Validation Failed: 99% GJR VaR must be >= 95% on every day."
assert (var_995 >= var_99).all(), "Validation Failed: 99.5% GJR VaR must be >= 99% on every day."
 
print("All mathematical constraints validated across the full out-of-sample period.")

All mathematical constraints validated across the full out-of-sample period.


In [24]:
out_of_sample_gjr.to_csv('gjr_garch_scaled_var_es.csv')
print("Saved: gjr_garch_scaled_var_es.csv")

Saved: gjr_garch_scaled_var_es.csv
